# Landing → Bronze
### Projeto CineData Analytics (base combinada TMDB/IMDb)

Objetivos deste notebook:
1. Criar o `catalog`, o database da camada Bronze e o **Volume** `inputs`, que recebe os 5 arquivos CSV da pasta Inputs (Landing)
2. Validar que os arquivos de origem estão disponíveis no Volume
3. Criar as tabelas Bronze a partir dos CSVs, sem nenhuma regra de negócio
4. Extrair a cotação do dólar via API do Banco Central

Regras da ingestão:
- Os CSVs são lidos **sem `inferSchema`**: todas as colunas chegam como texto, exatamente como estão na origem. A tipagem é responsabilidade da Silver.
- Cada tabela recebe a coluna `ingestion_datetime`, adicionada na mesma cadeia que termina em `.write()`. Por causa da Lazy Evaluation, o timestamp só é calculado no momento da escrita.
- A gravação usa o modo **append**: cada execução adiciona um novo lote, e a Silver fica responsável por manter apenas a versão mais recente de cada registro.

## 1. Parâmetros do ambiente

In [0]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "cinedata_analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

inputs_path = f"/Volumes/{catalog}/{bronze_schema_name}/inputs"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"inputs_path: {inputs_path}")

catalog: cinedata_analytics
bronze_schema: cinedata_analytics.bronze
silver_schema: cinedata_analytics.silver
gold_schema: cinedata_analytics.gold
inputs_path: /Volumes/cinedata_analytics/bronze/inputs


## 2. Criação do Catalog, do database Bronze e do Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.inputs")

DataFrame[]

## 3. Upload dos arquivos de origem

Os 5 arquivos foram enviados para a pasta **Inputs** em **Catalog** → `cinedata_analytics` → `bronze` → `inputs`
- `movies_info_TMDB_IMDB.csv`
- `movies_financials_IMDB_TMDB.csv`
- `movies_metrics_IMDB_TMDB.csv`
- `credits_and_tags_IMDB_TMDB.csv`
- `movies_reviews.csv`

## 4. Validação do Volume
Se algum arquivo estiver faltando, a execução é interrompida aqui, para que o Job não avance para as próximas camadas com dados incompletos.

In [0]:
expected_files = [
    "movies_info_TMDB_IMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_reviews.csv",
]

try:
    existing = {f.name for f in dbutils.fs.ls(inputs_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{inputs_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    raise FileNotFoundError(f"Arquivos não encontrados no volume: {', '.join(missing)}")
else:
    print("[OK] Todos os arquivos esperados estão no volume:")
    for f in dbutils.fs.ls(inputs_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")

[OK] Todos os arquivos esperados estão no volume:
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)


## 5. Tabelas Bronze a partir dos CSVs

In [0]:
from pyspark.sql.functions import current_timestamp

# Mapeamento dos caminhos no Volume
path_movies_info = f"{inputs_path}/movies_info_TMDB_IMDB.csv"
path_movies_financials = f"{inputs_path}/movies_financials_IMDB_TMDB.csv"
path_movies_metrics = f"{inputs_path}/movies_metrics_IMDB_TMDB.csv"
path_credits_and_tags = f"{inputs_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_reviews = f"{inputs_path}/movies_reviews.csv"

# Leitura pura (sem transformações)
df_movies_info_raw = spark.read.csv(path_movies_info, header=True)
df_movies_financials_raw = spark.read.csv(path_movies_financials, header=True)
df_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True)
df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True)
df_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True)

In [0]:
# Gravação com adição do timestamp no momento da escrita
df_movies_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_info")

df_movies_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_financials")

df_movies_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_metrics")

df_credits_and_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")

df_movies_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_reviews")

display(spark.table(f"{bronze_schema}.tb_movies_info").limit(5))

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-13T14:40:37.553Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-13T14:40:37.553Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-13T14:40:37.553Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-13T14:40:37.553Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-13T14:40:37.553Z


## 6. Ingestão via API · Cotação do dólar (PTAX)

A cotação de compra do dólar é extraída da API Olinda do Banco Central.
- As datas de início e fim são parâmetros do notebook (widgets) no formato `MM-DD-AAAA`.
- Por padrão consultamos os **últimos 7 dias corridos**, já que a API não retorna cotação em fins de semana e feriados.

In [0]:
from datetime import date, timedelta

dbutils.widgets.text("data_inicio", (date.today() - timedelta(days=7)).strftime("%m-%d-%Y"), "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", date.today().strftime("%m-%d-%Y"), "Data fim (MM-DD-AAAA)")

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

In [0]:
import requests
import pandas as pd

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

resposta = requests.get(url, timeout=30)
resposta.raise_for_status()
cotacoes = resposta.json()["value"]

if not cotacoes:
    raise ValueError("A API não retornou cotações para o período informado. Ajuste as datas nos widgets.")

df_cotacao_raw = spark.createDataFrame(pd.DataFrame(cotacoes))

df_cotacao_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

display(spark.table(f"{bronze_schema}.tb_cotacao_dolar").orderBy("dataHoraCotacao"))

cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.2198,2026-09-29 13:06:06.446305,2026-10-06T20:27:57.221Z
5.2198,2026-09-29 13:06:06.446305,2026-10-06T20:13:58.525Z
5.1803,2026-09-30 13:11:44.783262,2026-10-06T20:13:58.525Z
5.1803,2026-09-30 13:11:44.783262,2026-10-06T20:27:57.221Z
5.2073,2026-10-01 13:10:35.4469,2026-10-06T20:13:58.525Z
5.2073,2026-10-01 13:10:35.4469,2026-10-06T20:27:57.221Z
5.2232,2026-10-02 13:03:16.256632,2026-10-06T20:13:58.525Z
5.2232,2026-10-02 13:03:16.256632,2026-10-06T20:27:57.221Z
4.9853,2026-10-05 13:07:36.558602,2026-10-12T20:24:56.135Z
4.9853,2026-10-05 13:07:36.558602,2026-10-12T20:05:32.318Z
